In [4]:
source_csv = '../../en.openfoodfacts.org.products.csv'
source_parquet = '../../food.parquet'

print('TP 1:')
print()

import platform
print(f"Python version : {platform.python_version()}")

import pandas as pd
import numpy as np
import datetime as dt

print(f"Pandas version : {pd.__version__}")
print(f"Numpy version : {np.__version__}")

TP 1:

Python version : 3.12.10
Pandas version : 3.0.5
Numpy version : 2.5.2


In [5]:
# Outils

from pathlib import Path
import humanize

# Time
import time

class ExecutionTime:
    _start = 0

    def __init__(self):
        self._start = 0

    def start(self):
        self._start = time.time()

    def end(self, text = ""):
        time_exec = time.time() - self._start
        print(f"Execution time {text}: {humanize.precisedelta(time_exec, minimum_unit="microseconds")}")
        print('-----------------------\n')

t = ExecutionTime()

In [6]:
print('CSV:')
print()

t.start()
file_csv = Path(source_csv)
print(f"CSV size: {humanize.naturalsize(file_csv.stat().st_size)}")
print(f"Last metadata change: {dt.datetime.fromtimestamp((file_csv.stat().st_ctime)).strftime("%Y-%m-%d %H:%M:%S")}")
t.end("(pathlib.stat())")

print('CSV SAMPLE DATA')

t.start()

df_csv = pd.read_csv(
    source_csv,
    skipinitialspace=True,
    sep="\t",
    nrows=3,
)

print(f"Dimensions : {df_csv.shape}")
print(f"→ {df_csv.shape[0]} lines, {df_csv.shape[1]} columns\n")

print('Columns name:')
columns_name = list(df_csv)
display(list(df_csv))

print('Sample data:')
transposed = np.transpose(df_csv)
with pd.option_context('display.max_rows', None):
    display(transposed)

t.end()

CSV:

CSV size: 13.0 GB
Last metadata change: 2026-08-18 08:50:54
Execution time (pathlib.stat()): 1 millisecond and 448 microseconds
-----------------------

CSV SAMPLE DATA
Dimensions : (3, 211)
→ 3 lines, 211 columns

Columns name:


['code',
 'url',
 'creator',
 'created_t',
 'created_datetime',
 'last_modified_t',
 'last_modified_datetime',
 'last_modified_by',
 'last_updated_t',
 'last_updated_datetime',
 'product_name',
 'abbreviated_product_name',
 'generic_name',
 'quantity',
 'packaging',
 'packaging_tags',
 'packaging_en',
 'packaging_text',
 'brands',
 'brands_tags',
 'brands_en',
 'categories',
 'categories_tags',
 'categories_en',
 'origins',
 'origins_tags',
 'origins_en',
 'manufacturing_places',
 'manufacturing_places_tags',
 'labels',
 'labels_tags',
 'labels_en',
 'emb_codes',
 'emb_codes_tags',
 'first_packaging_code_geo',
 'cities',
 'cities_tags',
 'purchase_places',
 'stores',
 'countries',
 'countries_tags',
 'countries_en',
 'ingredients_text',
 'ingredients_tags',
 'ingredients_analysis_tags',
 'allergens',
 'allergens_en',
 'traces',
 'traces_tags',
 'traces_en',
 'serving_size',
 'serving_quantity',
 'no_nutrition_data',
 'additives_n',
 'additives',
 'additives_tags',
 'additives_en',
 'nu

Sample data:


,0,1,2
code,54,63,114
url,http://world-en.openfoodfacts.org/product/0000...,http://world-en.openfoodfacts.org/product/0000...,http://world-en.openfoodfacts.org/product/0000...
creator,kiliweb,kiliweb,kiliweb
created_t,1582569031,1673620307,1580066482
created_datetime,2020-02-24T18:30:31Z,2023-01-13T14:31:47Z,2020-01-26T19:21:22Z
last_modified_t,1733085204,1750061386,1751035658
last_modified_datetime,2024-12-01T20:33:24Z,2025-06-16T08:09:46Z,2025-06-27T14:47:38Z
last_modified_by,NaN,bodysupport,teolemon
last_updated_t,1740205422,1750061386,1751035658
last_updated_datetime,2025-02-22T06:23:42Z,2025-06-16T08:09:46Z,2025-06-27T14:47:38Z


Execution time : 52 milliseconds and 309 microseconds
-----------------------



In [7]:
print('CSV FULL DATA (filtrage colonnes utiles pour analyse)')

check_duplicated = ["code"]

use_cols = check_duplicated + [
    "product_name", "brands",
    "countries",
    "energy_100g", "sugars_100g", "salt_100g",
    "nutriscore_score"
]

t.start()

chunk_iterator_csv = pd.read_csv(
    source_csv,
    memory_map=True,
    skipinitialspace=True,
    sep="\t",
    chunksize=10000,
    low_memory = False,
    on_bad_lines="skip",
    usecols=use_cols
)

full_data_csv = []

for chunk in chunk_iterator_csv:
    full_data_csv.append(chunk)

df_full_csv = pd.concat(full_data_csv, ignore_index=True)

t.end('Load data')

CSV FULL DATA (filtrage colonnes utiles pour analyse)
Execution time Load data: 1 minute, 30 seconds, 544 milliseconds and 342 microseconds
-----------------------



In [8]:
print('Analyse:')

t.start()

display(df_full_csv.info())

print(f"Dimensions: {df_full_csv.shape}")
print(f"→ {df_full_csv.shape[0]} lines, {df_full_csv.shape[1]} columns")
print()

print(f"Duplication: {check_duplicated}:")
print(df_full_csv.duplicated(subset=check_duplicated).value_counts())
print()

print("Valeurs non-renseignées | Taux de remplissage par colonnes:")
missing = pd.DataFrame({
    'total_manquants': df_full_csv.isna().sum(),
    '%': 100 - (df_full_csv.isna().sum() / len(df_full_csv) * 100).round(2)
})
print(missing)
print()

# Voir https://static.openfoodfacts.org/data/data-fields.txt
print("Produits vendu en France ([countries].str.contains(fr)):")
print(df_full_csv[df_full_csv["countries"].str.contains("fr")].shape[0])
print()

print("Top 10 marques: ")
print(df_full_csv.groupby(["brands"]).size().sort_values(ascending=False)[0:10])
print()

print("Quelle part de Nutri-Score renseigné ?")
print(f"Total: {df_full_csv.shape[0]}")
print(f"Manquant: {df_full_csv["nutriscore_score"].isna().sum()}")
print(f"Nutri-Score non-renseignés: {(df_full_csv["nutriscore_score"].isna().sum() / len(df_full_csv) * 100).round(2)}%")

t.end()

Analyse:
<class 'pandas.DataFrame'>
RangeIndex: 4532767 entries, 0 to 4532766
Data columns (total 8 columns):
 #   Column            Dtype  
---  ------            -----  
 0   code              object 
 1   product_name      str    
 2   brands            str    
 3   countries         str    
 4   nutriscore_score  object 
 5   energy_100g       object 
 6   sugars_100g       float64
 7   salt_100g         float64
dtypes: float64(2), object(3), str(3)
memory usage: 434.5+ MB


None

Dimensions: (4532767, 8)
→ 4532767 lines, 8 columns

Duplication: ['code']:
False    4532701
True          66
Name: count, dtype: int64

Valeurs non-renseignées | Taux de remplissage par colonnes:
                  total_manquants       %
code                            0  100.00
product_name               335833   92.59
brands                    1682789   62.88
countries                   22828   99.50
nutriscore_score          3157914   30.33
energy_100g               2296811   49.33
sugars_100g               2388528   47.31
salt_100g                 2580977   43.06

Produits vendu en France ([countries].str.contains(fr)):
590241

Top 10 marques: 
brands
Carrefour    25641
Auchan       18107
Coop         14067
Lidl         13804
U            12286
BonÀrea      12193
Aldi         11683
Hacendado    10490
Tesco        10282
Delhaize      9741
dtype: int64

Quelle part de Nutri-Score renseigné ?
Total: 4532767
Manquant: 3157914
Nutri-Score non-renseignés: 69.67%
Execution time : 3 secon

In [9]:
print('PARQUET:')
print()

import pyarrow as pa
import duckdb as db

print(f"Pyarrow version : {pa.__version__}")
print(f"Duckdb version : {db.__version__}")

PARQUET:

Pyarrow version : 25.0.1
Duckdb version : 1.5.5


In [10]:
file_parquet = Path(source_parquet)

print()
print(f"Parquet size: {humanize.naturalsize(file_parquet.stat().st_size)}")
print(f"Last metadata change: {dt.datetime.fromtimestamp((file_parquet.stat().st_ctime)).strftime("%Y-%m-%d %H:%M:%S")}")
t.end("(pathlib.stat())")


Parquet size: 7.7 GB
Last metadata change: 2026-07-29 14:51:32
Execution time (pathlib.stat()): 3 seconds, 239 milliseconds and 598 microseconds
-----------------------



In [11]:
import pyarrow.parquet as pq

schema = pq.read_schema(source_parquet)

print('Columns name:')

schema_columns_name = []

for field in schema:
    schema_columns_name.append(field.name)
    print(field.name, field.type)

Columns name:
additives_n int32
additives_tags list<element: string>
allergens_tags list<element: string>
brands_tags list<element: string>
brands string
categories string
categories_tags list<element: string>
categories_properties struct<ciqual_food_code: int32, agribalyse_food_code: int32, agribalyse_proxy_food_code: int32>
checkers_tags list<element: string>
ciqual_food_name_tags list<element: string>
cities_tags list<element: string>
code string
compared_to_category string
complete int32
completeness float
correctors_tags list<element: string>
countries_tags list<element: string>
created_t int64
creator string
data_quality_errors_tags list<element: string>
data_quality_info_tags list<element: string>
data_quality_warnings_tags list<element: string>
data_sources_tags list<element: string>
environmental_score_data string
environmental_score_grade string
environmental_score_score int32
environmental_score_tags list<element: string>
editors list<element: string>
emb_codes_tags list<ele

In [12]:
print('LOAD PARQUET FILE')
print()

check_duplicated = ["code"]

use_cols = check_duplicated + [
    "product_name", "brands",
    "lang",
    "nutriscore_score"
]

t.start()

df_parquet = pd.read_parquet(
    source_parquet,
    engine="pyarrow",
    columns=use_cols
)

display(df_parquet.info())

print(f"Dimensions : {df_parquet.shape}")
print(f"→ {df_parquet.shape[0]} lines, {df_parquet.shape[1]} columns\n")

t.end()

LOAD PARQUET FILE

<class 'pandas.DataFrame'>
RangeIndex: 4636471 entries, 0 to 4636470
Data columns (total 5 columns):
 #   Column            Dtype  
---  ------            -----  
 0   code              str    
 1   product_name      object 
 2   brands            str    
 3   lang              str    
 4   nutriscore_score  float64
dtypes: float64(1), object(1), str(3)
memory usage: 273.2+ MB


None

Dimensions : (4636471, 5)
→ 4636471 lines, 5 columns

Execution time : 10 seconds, 675 milliseconds and 596 microseconds
-----------------------



In [13]:
print('PARQUET SAMPLE DATA')

t.start()

with pd.option_context('display.max_colwidth', None):
    display(df_parquet.head(3))

t.end()

PARQUET SAMPLE DATA


,code,product_name,brands,lang,nutriscore_score
0,0000101209159,"[{'lang': 'main', 'text': 'Véritable pâte à tartiner noisettes chocolat noir'}, {'lang': 'fr', 'text': 'Véritable pâte à tartiner noisettes chocolat noir'}]",Bovetti,fr,25.0
1,0000105000011,"[{'lang': 'main', 'text': 'Chamomile Herbal Tea'}, {'lang': 'en', 'text': 'Chamomile Herbal Tea'}]",Lagg's,en,NaN
2,0000105000042,"[{'lang': 'main', 'text': 'Lagg's, herbal tea, peppermint'}, {'lang': 'en', 'text': 'Lagg's, herbal tea, peppermint'}]",Lagg's,en,NaN


Execution time : 13 milliseconds and 132 microseconds
-----------------------



In [14]:
t.start()

print(f"Duplication {check_duplicated}:")
print(df_parquet.duplicated(subset=check_duplicated).value_counts())
print()

print("Valeurs non-renseignées | Taux de remplissage par colonnes:")
missing = pd.DataFrame({
    'total_manquants': df_parquet.isna().sum(),
    '%': 100 - (df_parquet.isna().sum() / len(df_parquet) * 100).round(2)
})
print(missing)
print()

print("Produits vendu en France ([lang].eq(fr):)") #count
print(df_parquet[df_parquet["lang"].eq("fr")].shape[0])
print()

print("Top 10 marques: ")
print(df_parquet.groupby(["brands"]).size().sort_values(ascending=False)[0:10])
print()

print("Quelle part a un Nutri-Score renseigné ?")
print(f"Total: {df_parquet.shape[0]}")
print(f"Manquant: {df_parquet["nutriscore_score"].isna().sum()}")
print(f"Nutri-Score non-renseignés: {(df_parquet["nutriscore_score"].isna().sum() / len(df_full_csv) * 100).round(2)}%")

t.end()

Duplication ['code']:
False    4636411
True          60
Name: count, dtype: int64

Valeurs non-renseignées | Taux de remplissage par colonnes:
                  total_manquants       %
code                            0  100.00
product_name                    0  100.00
brands                    1595058   65.60
lang                            4  100.00
nutriscore_score          3255402   29.79

Produits vendu en France ([lang].eq(fr):)
1336912

Top 10 marques: 
brands
             108709
Carrefour     20732
Coop          14545
Lidl          14243
U             12384
Aldi          12353
BonÀrea       12155
Hacendado     10658
Auchan        10590
Tesco         10503
dtype: int64

Quelle part a un Nutri-Score renseigné ?
Total: 4636471
Manquant: 3255402
Nutri-Score non-renseignés: 71.82%
Execution time : 2 seconds, 731 milliseconds and 75 microseconds
-----------------------

